# Manejo de la BD de Firebase Firestore
Este cuaderno se genera con la necesidad de crear un catálogo de productos e insertarlos en Firebase Firestore.

## Conexión con Firebase

In [3]:
#Uso de Firebase Firestore
import firebase_admin
from firebase_admin import credentials
from firebase_admin import firestore

#Sistema
import os
from dotenv import load_dotenv

In [5]:
# Carga las variables del archivo .env a las variables de entorno
load_dotenv()

try:
# Recupera la ruta al archivo de credenciales
    cred_path = os.getenv("FIREBASE_CREDENTIALS_PATH")

    if cred_path:
        cred = credentials.Certificate(cred_path)
        firebase_admin.initialize_app(cred)
        print("¡Firebase conectado de manera segura!")
    else:
        print("Error: No se encontró la ruta de las credenciales en el archivo .env")
except Exception as e:
    print(f"Error al inicializar Firebase: {e}")


Error al inicializar Firebase: The default Firebase app already exists. This means you called initialize_app() more than once without providing an app name as the second argument. In most cases you only need to call initialize_app() once. But if you do want to initialize multiple apps, pass a second argument to initialize_app() to give each app a unique name.


## Añadiendo productos

Se espera que cada documento en la colección `productos` tenga estos campos:

| Atributo | Tipo | Descripción | Ejemplo |
|:--------:|:----:|:-----------:|:-------:|
|`sku`| String | Id interno único | `"PIS-MST-35-750"`|
|`nombre`| String | Nombre comercial estándar | `"Pisco Mistral 35° Especial"`|
|`marca`| String | Marca del fabricante | `"Mistral"`|
|`categoria`| String | Rubro principal | `"Destilados", "Cervezas", "Bebidas"`|
|`subcategoria`| String | Tipo específico de licor | `"Pisco", "Ron", "Lager"`|
|`formato`| String | Presentación o calibre | `"750cc", "1.5L", "Pack 6 x 355cc"`|
|`codigo_barra`| String | Código de envase | `"7804320123456"`|
|`precio_costo`| Number | Costo neto de compra (Factura) | `5500`|
|`precio_venta`| Number | Precio de venta al consumidor | `7500`|
|`stock_actual`| Number | Unidades disponibles | `20`|
|`stock_minimo`| Number | Umbral de alerta para reponer | `6`|
|`activo`| Boolean | Descontinuar producto sin borrar | `true`|
|`creado_en`| Timestamp | Marca de tiempo cuando se agrega | `firestore.SERVER_TIMESTAMP`|

### Catálogo inicial de productos

Se crea un catálogo de 40 productos para la botillería.

In [ ]:
db = firestore.client()
productos_iniciales = [
  {"sku": "PIS-CAP-30-750", "nombre": "Pisco Capel 30°", "marca": "Capel", "categoria": "Destilados", "subcategoria": "Pisco", "formato": "750cc", "codigo_barra": "7802820001012", "precio_costo": 3200, "precio_venta": 5490, "stock_actual": 48, "stock_minimo": 15, "activo": True},
  {"sku": "PIS-ADC-40-750", "nombre": "Pisco Alto del Carmen 40°", "marca": "Alto del Carmen", "categoria": "Destilados", "subcategoria": "Pisco", "formato": "750cc", "codigo_barra": "7802820001029", "precio_costo": 5100, "precio_venta": 8490, "stock_actual": 30, "stock_minimo": 10, "activo": True},
  {"sku": "PIS-MST-40-750", "nombre": "Pisco Mistral 40° Reservado", "marca": "Mistral", "categoria": "Destilados", "subcategoria": "Pisco", "formato": "750cc", "codigo_barra": "7804320001033", "precio_costo": 4800, "precio_venta": 7990, "stock_actual": 25, "stock_minimo": 10, "activo": True},
  {"sku": "PIS-HQE-35-750", "nombre": "Pisco Horcón Quemado Especial 35°", "marca": "Horcón Quemado", "categoria": "Destilados", "subcategoria": "Pisco", "formato": "750cc", "codigo_barra": "7802820001043", "precio_costo": 3600, "precio_venta": 5990, "stock_actual": 20, "stock_minimo": 8, "activo": True},
  {"sku": "RON-BAC-BLC-750", "nombre": "Ron Bacardi Carta Blanca", "marca": "Bacardi", "categoria": "Destilados", "subcategoria": "Ron", "formato": "750cc", "codigo_barra": "7802820001050", "precio_costo": 6200, "precio_venta": 9990, "stock_actual": 18, "stock_minimo": 8, "activo": True},
  {"sku": "RON-HAV-A3-750", "nombre": "Ron Havana Club Añejo 3 Años", "marca": "Havana Club", "categoria": "Destilados", "subcategoria": "Ron", "formato": "750cc", "codigo_barra": "7802820001067", "precio_costo": 7000, "precio_venta": 11490, "stock_actual": 14, "stock_minimo": 6, "activo": True},
  {"sku": "WHI-JWR-750", "nombre": "Whisky Johnnie Walker Red Label", "marca": "Johnnie Walker", "categoria": "Destilados", "subcategoria": "Whisky", "formato": "750cc", "codigo_barra": "5000267023655", "precio_costo": 9500, "precio_venta": 14990, "stock_actual": 22, "stock_minimo": 8, "activo": True},
  {"sku": "WHI-OPR-12-750", "nombre": "Whisky Old Parr 12 Años", "marca": "Old Parr", "categoria": "Destilados", "subcategoria": "Whisky", "formato": "750cc", "codigo_barra": "5000267166642", "precio_costo": 15800, "precio_venta": 23990, "stock_actual": 10, "stock_minimo": 5, "activo": True},
  {"sku": "WHI-BUC-DLX-750", "nombre": "Whisky Buchanan's Deluxe", "marca": "Buchanan's", "categoria": "Destilados", "subcategoria": "Whisky", "formato": "750cc", "codigo_barra": "5000267166895", "precio_costo": 13200, "precio_venta": 19990, "stock_actual": 12, "stock_minimo": 5, "activo": True},
  {"sku": "VOD-ABS-ORI-750", "nombre": "Vodka Absolut Original", "marca": "Absolut", "categoria": "Destilados", "subcategoria": "Vodka", "formato": "750cc", "codigo_barra": "7312040017005", "precio_costo": 8200, "precio_venta": 12990, "stock_actual": 20, "stock_minimo": 8, "activo": True},
  {"sku": "VOD-SMI-RED-750", "nombre": "Vodka Smirnoff Red", "marca": "Smirnoff", "categoria": "Destilados", "subcategoria": "Vodka", "formato": "750cc", "codigo_barra": "5410316911015", "precio_costo": 5800, "precio_venta": 8990, "stock_actual": 26, "stock_minimo": 10, "activo": True},
  {"sku": "GIN-BFT-LDR-750", "nombre": "Gin Beefeater London Dry", "marca": "Beefeater", "categoria": "Destilados", "subcategoria": "Gin", "formato": "750cc", "codigo_barra": "5010196120067", "precio_costo": 8900, "precio_venta": 13990, "stock_actual": 15, "stock_minimo": 6, "activo": True},
  {"sku": "GIN-BOM-SAP-750", "nombre": "Gin Bombay Sapphire", "marca": "Bombay Sapphire", "categoria": "Destilados", "subcategoria": "Gin", "formato": "750cc", "codigo_barra": "5000299601403", "precio_costo": 10200, "precio_venta": 15990, "stock_actual": 12, "stock_minimo": 6, "activo": True},
  {"sku": "TEQ-JCE-750", "nombre": "Tequila Jose Cuervo Especial", "marca": "Jose Cuervo", "categoria": "Destilados", "subcategoria": "Tequila", "formato": "750cc", "codigo_barra": "7501035106017", "precio_costo": 9600, "precio_venta": 14990, "stock_actual": 14, "stock_minimo": 6, "activo": True},
  {"sku": "TEQ-1800-SLV-750", "nombre": "Tequila 1800 Silver", "marca": "1800", "categoria": "Destilados", "subcategoria": "Tequila", "formato": "750cc", "codigo_barra": "7501035101142", "precio_costo": 13500, "precio_venta": 20990, "stock_actual": 8, "stock_minimo": 4, "activo": True},
  {"sku": "CER-CRI-LAG-350", "nombre": "Cerveza Cristal Lager Lata", "marca": "Cristal", "categoria": "Cervezas", "subcategoria": "Lager", "formato": "350cc", "codigo_barra": "7802900001013", "precio_costo": 550, "precio_venta": 990, "stock_actual": 240, "stock_minimo": 60, "activo": True},
  {"sku": "CER-ESC-LAG-350", "nombre": "Cerveza Escudo", "marca": "Escudo", "categoria": "Cervezas", "subcategoria": "Lager", "formato": "350cc", "codigo_barra": "7802900001020", "precio_costo": 580, "precio_venta": 1050, "stock_actual": 200, "stock_minimo": 60, "activo": True},
  {"sku": "CER-KUN-TOR-500", "nombre": "Cerveza Kunstmann Torobayo", "marca": "Kunstmann", "categoria": "Cervezas", "subcategoria": "Lager", "formato": "500cc", "codigo_barra": "7802900001037", "precio_costo": 1100, "precio_venta": 1890, "stock_actual": 96, "stock_minimo": 24, "activo": True},
  {"sku": "CER-HEI-350", "nombre": "Cerveza Heineken", "marca": "Heineken", "categoria": "Cervezas", "subcategoria": "Lager", "formato": "350cc", "codigo_barra": "8712000021621", "precio_costo": 900, "precio_venta": 1590, "stock_actual": 150, "stock_minimo": 40, "activo": True},
  {"sku": "CER-COR-EXT-355", "nombre": "Cerveza Corona Extra", "marca": "Corona", "categoria": "Cervezas", "subcategoria": "Lager", "formato": "355cc", "codigo_barra": "7501039101615", "precio_costo": 1000, "precio_venta": 1790, "stock_actual": 120, "stock_minimo": 30, "activo": True},
  {"sku": "CER-AUS-CAL-330", "nombre": "Cerveza Austral Calafate", "marca": "Austral", "categoria": "Cervezas", "subcategoria": "Ale", "formato": "330cc", "codigo_barra": "7802900001044", "precio_costo": 1050, "precio_venta": 1790, "stock_actual": 72, "stock_minimo": 24, "activo": True},
  {"sku": "VIN-CDD-CAB-750", "nombre": "Vino Casillero del Diablo Cabernet Sauvignon", "marca": "Casillero del Diablo", "categoria": "Vinos", "subcategoria": "Tinto", "formato": "750cc", "codigo_barra": "7804300001010", "precio_costo": 3200, "precio_venta": 5490, "stock_actual": 60, "stock_minimo": 20, "activo": True},
  {"sku": "VIN-SR120-MER-750", "nombre": "Vino Santa Rita 120 Merlot", "marca": "Santa Rita", "categoria": "Vinos", "subcategoria": "Tinto", "formato": "750cc", "codigo_barra": "7804300001027", "precio_costo": 2600, "precio_venta": 4490, "stock_actual": 55, "stock_minimo": 20, "activo": True},
  {"sku": "VIN-FRO-SBL-750", "nombre": "Vino Concha y Toro Frontera Sauvignon Blanc", "marca": "Frontera", "categoria": "Vinos", "subcategoria": "Blanco", "formato": "750cc", "codigo_barra": "7804300001034", "precio_costo": 2200, "precio_venta": 3990, "stock_actual": 50, "stock_minimo": 20, "activo": True},
  {"sku": "VIN-UND-SIB-CAR-750", "nombre": "Vino Undurraga Sibaris Carmenere", "marca": "Undurraga", "categoria": "Vinos", "subcategoria": "Tinto", "formato": "750cc", "codigo_barra": "7804300001041", "precio_costo": 4100, "precio_venta": 6990, "stock_actual": 30, "stock_minimo": 10, "activo": True},
  {"sku": "ESP-VAL-BRT-750", "nombre": "Espumante Valdivieso Brut", "marca": "Valdivieso", "categoria": "Vinos", "subcategoria": "Espumante", "formato": "750cc", "codigo_barra": "7804300001058", "precio_costo": 3400, "precio_venta": 5990, "stock_actual": 40, "stock_minimo": 15, "activo": True},
  {"sku": "CHA-CHD-BRT-750", "nombre": "Champagne Chandon Brut", "marca": "Chandon", "categoria": "Vinos", "subcategoria": "Champagne", "formato": "750cc", "codigo_barra": "7804300001065", "precio_costo": 7800, "precio_venta": 12990, "stock_actual": 18, "stock_minimo": 6, "activo": True},
  {"sku": "LIC-BAI-ICR-750", "nombre": "Licor Baileys Irish Cream", "marca": "Baileys", "categoria": "Destilados", "subcategoria": "Licor", "formato": "750cc", "codigo_barra": "5011013101018", "precio_costo": 9200, "precio_venta": 14490, "stock_actual": 16, "stock_minimo": 6, "activo": True},
  {"sku": "LIC-JAG-700", "nombre": "Licor Jägermeister", "marca": "Jägermeister", "categoria": "Destilados", "subcategoria": "Licor", "formato": "700cc", "codigo_barra": "4102300010109", "precio_costo": 9800, "precio_venta": 15990, "stock_actual": 14, "stock_minimo": 5, "activo": True},
  {"sku": "APE-CAM-750", "nombre": "Aperitivo Campari", "marca": "Campari", "categoria": "Destilados", "subcategoria": "Aperitivo", "formato": "750cc", "codigo_barra": "8002270004017", "precio_costo": 8600, "precio_venta": 13490, "stock_actual": 12, "stock_minimo": 5, "activo": True},
  {"sku": "APE-MRT-ROS-750", "nombre": "Aperitivo Martini Rosso", "marca": "Martini", "categoria": "Destilados", "subcategoria": "Vermut", "formato": "750cc", "codigo_barra": "8000350040014", "precio_costo": 5600, "precio_venta": 8990, "stock_actual": 20, "stock_minimo": 8, "activo": True},
  {"sku": "VIN-BOR-TIN-1000", "nombre": "Vino Reservado Tinto Caja", "marca": "Reservado", "categoria": "Vinos", "subcategoria": "Borgoña", "formato": "1L", "codigo_barra": "7804300001072", "precio_costo": 1500, "precio_venta": 2490, "stock_actual": 45, "stock_minimo": 15, "activo": True},
  {"sku": "BEB-COC-1500", "nombre": "Bebida Coca-Cola Original", "marca": "Coca-Cola", "categoria": "No Alcohólicos", "subcategoria": "Bebida gaseosa", "formato": "1.5L", "codigo_barra": "7802820002019", "precio_costo": 1200, "precio_venta": 1990, "stock_actual": 80, "stock_minimo": 24, "activo": True},
  {"sku": "BEB-SPR-1500", "nombre": "Bebida Sprite", "marca": "Sprite", "categoria": "No Alcohólicos", "subcategoria": "Bebida gaseosa", "formato": "1.5L", "codigo_barra": "7802820002026", "precio_costo": 1150, "precio_venta": 1890, "stock_actual": 60, "stock_minimo": 20, "activo": True},
  {"sku": "BEB-FAN-1500", "nombre": "Bebida Fanta Naranja", "marca": "Fanta", "categoria": "No Alcohólicos", "subcategoria": "Bebida gaseosa", "formato": "1.5L", "codigo_barra": "7802820002033", "precio_costo": 1150, "precio_venta": 1890, "stock_actual": 60, "stock_minimo": 20, "activo": True},
  {"sku": "AGU-CAC-CGS-1500", "nombre": "Agua Mineral Cachantún Con Gas", "marca": "Cachantún", "categoria": "No Alcohólicos", "subcategoria": "Agua mineral", "formato": "1.5L", "codigo_barra": "7802820002040", "precio_costo": 700, "precio_venta": 1290, "stock_actual": 70, "stock_minimo": 20, "activo": True},
  {"sku": "ENE-RDB-250", "nombre": "Bebida Energética Red Bull", "marca": "Red Bull", "categoria": "No Alcohólicos", "subcategoria": "Energética", "formato": "250cc", "codigo_barra": "9002490100070", "precio_costo": 1400, "precio_venta": 2390, "stock_actual": 90, "stock_minimo": 24, "activo": True},
  {"sku": "JUG-DVA-NAR-1000", "nombre": "Jugo Andina Del Valle Naranja", "marca": "Del Valle", "categoria": "No Alcohólicos", "subcategoria": "Jugo", "formato": "1L", "codigo_barra": "7802820002057", "precio_costo": 1300, "precio_venta": 2190, "stock_actual": 40, "stock_minimo": 12, "activo": True},
  {"sku": "HIE-CUB-2K", "nombre": "Hielo en Cubos", "marca": "Genérico", "categoria": "Otros", "subcategoria": "Hielo", "formato": "2kg", "codigo_barra": "7802820002064", "precio_costo": 700, "precio_venta": 1490, "stock_actual": 35, "stock_minimo": 10, "activo": True},
  {"sku": "SNK-LAY-150", "nombre": "Papas Fritas Lays Original", "marca": "Lays", "categoria": "Otros", "subcategoria": "Snacks", "formato": "150g", "codigo_barra": "7802820002071", "precio_costo": 1300, "precio_venta": 2190, "stock_actual": 50, "stock_minimo": 15, "activo": False}
]

### Insert de productos
Se crean las funciones necesarias para insertar el catálogo entero o un producto seleccionado en formato .JSON

In [11]:
def insertar_catalogo(productos):
    coleccion = db.collection('productos')
    batch = db.batch()
    for producto in productos:
        doc_ref = coleccion.document(producto['sku'])
        
        datos = producto.copy()
        datos["creado_en"] = firestore.SERVER_TIMESTAMP
        datos["actualizado_en"] = firestore.SERVER_TIMESTAMP
        
        batch.set(doc_ref, datos, merge=True)
        
    batch.commit()
    print(f"Se han insertado/actualizado {len(productos)} productos en Firestore.")
    
    
def insertar_producto_individual(datos_producto):
    #Agregar verificacion que sku no exista en la coleccion antes de insertar
    coleccion = db.collection('productos')
    doc_ref = coleccion.document(datos_producto['sku'])
    
    # Verificar si el documento ya existe
    if doc_ref.get().exists:
        print(f"El producto con SKU {datos_producto['sku']} ya existe. No se insertará.")
        return
    
    # Insertar el producto
    sku = datos_producto.get("sku")
    if not sku:
        raise ValueError("El producto debe tener un SKU definido.")

    payload = datos_producto.copy()
    payload["creado_en"] = firestore.SERVER_TIMESTAMP
    payload["actualizado_en"] = firestore.SERVER_TIMESTAMP

    # merge=True evita sobreescribir datos si el documento ya existía
    doc_ref.set(payload, merge=True)
    print(f"Producto '{datos_producto['nombre']}' registrado con ID {sku}.")

In [12]:
#Insertando el catalogo inicial de productos en Firestore
insertar_catalogo(productos_iniciales)

Se han insertado/actualizado 40 productos en Firestore.


## Visualizar productos
Se crea una función para visualizar los productos en la BD.

In [13]:
# Visualizar productos
def visualizar_productos():
    coleccion = db.collection('productos')
    productos = coleccion.stream()
    
    for producto in productos:
        print(f"SKU: {producto.id}, Datos: {producto.to_dict()}")

In [14]:
visualizar_productos()

SKU: AGU-CAC-CGS-1500, Datos: {'codigo_barra': '7802820002040', 'actualizado_en': DatetimeWithNanoseconds(2026, 9, 9, 1, 52, 55, 148000, tzinfo=datetime.timezone.utc), 'nombre': 'Agua Mineral Cachantún Con Gas', 'categoria': 'No Alcohólicos', 'marca': 'Cachantún', 'precio_venta': 1290, 'formato': '1.5L', 'activo': True, 'stock_minimo': 20, 'sku': 'AGU-CAC-CGS-1500', 'precio_costo': 700, 'subcategoria': 'Agua mineral', 'creado_en': DatetimeWithNanoseconds(2026, 9, 9, 1, 52, 55, 148000, tzinfo=datetime.timezone.utc), 'stock_actual': 70}
SKU: APE-CAM-750, Datos: {'codigo_barra': '8002270004017', 'actualizado_en': DatetimeWithNanoseconds(2026, 9, 9, 1, 52, 55, 148000, tzinfo=datetime.timezone.utc), 'nombre': 'Aperitivo Campari', 'categoria': 'Destilados', 'marca': 'Campari', 'precio_venta': 13490, 'formato': '750cc', 'activo': True, 'stock_minimo': 5, 'sku': 'APE-CAM-750', 'precio_costo': 8600, 'subcategoria': 'Aperitivo', 'creado_en': DatetimeWithNanoseconds(2026, 9, 9, 1, 52, 55, 148000,